# Pillar 2z — Policy distillation on V12 (Colab A100/L4)

Train pillar2z by warm-starting from pillar2y2_epoch_40 on the V12 corpus (~7.4M states: 450 self-play games + ~9700 crisis recovery + ~9700 crisis prevention replays).

**Per Pillar 2u (HISTORY 110+):** policy-only training, `val_weight=0`. The model has no value head; we don't predict score. Each state's target is the MCTS visit distribution from a 400-sim NN-MCTS search.

**Per Phase 3R (HISTORY 133-136):** the player that generated V12 was pillar2y2 + value_head_v11 + q=2.0 (Phase 3R: mean 13,476 / median 16,427 cap-pinned).

**Settings:**
- 10 blocks × 256ch (matches pillar2y2)
- 60 epochs (pillar2y2 was still learning at 40)
- batch 4096, lr 1e-3, warmup 3 epochs
- AMP + torch.compile for throughput
- Warm-start from pillar2y2_epoch_40 (don't restart optimizer)
- Save every epoch to Drive (per `feedback_colab_checkpoints` memory)

**Upload to `MyDrive/alphatrain/` before running:**
1. `colorlines_selfplay_train.tar.gz` — code
2. `v12_pillar2z.pt` — the V12 training tensor (built locally on M5 by `build_expert_v2_tensor.py`)
3. `pillar2y2_epoch_40.pt` — warm-start checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil_t = time.time()
!cp {DRIVE}/v12_pillar2z.pt /content/alphatrain/data/v12_pillar2z.pt
print(f'Tensor: {os.path.getsize("/content/alphatrain/data/v12_pillar2z.pt")/1e9:.1f} GB ({time.time()-shutil_t:.0f}s)')

!cp {DRIVE}/pillar2y2_epoch_40.pt /content/alphatrain/data/pillar2y2_epoch_40.pt
print(f'Warm-start ckpt: {os.path.getsize("/content/alphatrain/data/pillar2y2_epoch_40.pt")/1e6:.0f} MB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === Pillar 2z Training ===
EPOCHS = 60
BATCH_SIZE = 4096
LR = 1e-3
WARMUP = 3
WD = 1e-4
SAVE_DIR = '/content/checkpoints/pillar2z'
DRIVE_SAVE_DIR = f'{DRIVE}/checkpoints_pillar2z'
# ==========================

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Local checkpoint dir: {SAVE_DIR}')
print(f'Drive checkpoint dir: {DRIVE_SAVE_DIR}')

%cd /content
!python -m alphatrain.train \
    --tensor-file alphatrain/data/v12_pillar2z.pt \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --lr {LR} --warmup-epochs {WARMUP} --weight-decay {WD} \
    --num-blocks 10 --channels 256 \
    --val-split 0.05 \
    --amp --compile \
    --resume alphatrain/data/pillar2y2_epoch_40.pt \
    --warm-start \
    --save-dir {SAVE_DIR} \
    --copy-to {DRIVE_SAVE_DIR} \
    --policy-only

## Notes

- **Resume support:** if Colab disconnects mid-training, change `--warm-start` to `--resume {DRIVE_SAVE_DIR}/best.pt` to continue from the last saved epoch with optimizer state.
- **Per-epoch save:** `--copy-to` writes every epoch's checkpoint to Drive — never lose progress (`feedback_colab_checkpoints`).
- **Wall time:**
  - A100: ~10-15 min/epoch × 60 = ~10-15h
  - L4: ~25-40 min/epoch × 60 = ~24-40h
  - T4: don't bother for full run (~3+ days)
- **Watch:** training loss should decrease through epoch 60. Per pillar2y2 history, val loss kept decreasing past 40 epochs, so 60 is the new target.
- **Sanity-check value-distribution before training** (`feedback_sanity_check_data` memory) — the data builder reports score histogram. Look for the "Mid-Game Blob" pattern (HISTORY lesson 24-28) — if 80%+ of samples cluster in one score bucket, training will saturate. Should be OK with V12 since most games are cap-pinned at ~16,400.
- **After training:** download `best.pt` and `epoch_60.pt` (or whichever is best by val-loss) from Drive. Eval locally:
  ```
  python -m alphatrain.scripts.eval_parallel \
    --model alphatrain/data/pillar2z_best.pt \
    --value-head-path alphatrain/data/value_head_v11.pt \
    --simulations 400 --q-weight 2.0 --workers 16 --device mps \
    --max-turns 8000 --mcts-only --early-stop \
    --seeds 0 1 2 3 ... 49
  ```